In [39]:
!pip install pyspark

In [40]:
from google.colab import files

uploaded = files.upload()

Saving data.zip to data (1).zip


In [41]:
#Version Environment & Dependencies
import pyspark
import pandas as pd
import platform

print("Python:", platform.python_version())
print("PySpark:", pyspark.__version__)
print("Pandas:", pd.__version__)

Python: 3.12.13
PySpark: 4.0.3
Pandas: 2.2.2


In [42]:
#Data Unzipping & Automated Extraction
import zipfile
import os

with zipfile.ZipFile("data.zip", "r") as zip_ref:
    zip_ref.extractall("data")

print("Files extracted successfully!")

Files extracted successfully!


In [43]:
import os

for root, dirs, files in os.walk("data"):
    for file in files:
        print(os.path.join(root, file))

data/data_group_1.csv
data/data_group_2.csv
data/data_group_3.csv
data/__MACOSX/._data_group_2.csv
data/__MACOSX/._data_group_1.csv
data/__MACOSX/._data_group_3.csv


In [44]:
#Spark Session Initialization
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Wind Turbine Pipeline") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [45]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/*.csv")

In [46]:
#Data Profiling and Overview
print("Total Records:", df.count())
print("Total Turbines:", df.select("turbine_id").distinct().count())
print("Columns:", df.columns)

Total Records: 11160
Total Turbines: 15
Columns: ['timestamp', 'turbine_id', 'wind_speed', 'wind_direction', 'power_output']


In [47]:
#Missing Value Inspection
from pyspark.sql.functions import *

missing = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing.show()

+---------+----------+----------+--------------+------------+
|timestamp|turbine_id|wind_speed|wind_direction|power_output|
+---------+----------+----------+--------------+------------+
|        0|         0|         0|             0|           0|
+---------+----------+----------+--------------+------------+



In [48]:
df.show(10)

+-------------------+----------+----------+--------------+------------+
|          timestamp|turbine_id|wind_speed|wind_direction|power_output|
+-------------------+----------+----------+--------------+------------+
|2022-03-01 00:00:00|        11|       9.1|           269|         2.9|
|2022-03-01 00:00:00|        12|      11.3|           316|         2.5|
|2022-03-01 00:00:00|        13|      11.2|           148|         3.7|
|2022-03-01 00:00:00|        14|      10.7|            97|         1.6|
|2022-03-01 00:00:00|        15|      11.0|            81|         4.4|
|2022-03-01 01:00:00|        11|      12.3|           245|         1.8|
|2022-03-01 01:00:00|        12|      11.0|           293|         2.2|
|2022-03-01 01:00:00|        13|      11.4|           270|         1.9|
|2022-03-01 01:00:00|        14|      10.4|           140|         2.3|
|2022-03-01 01:00:00|        15|      14.6|           283|         4.3|
+-------------------+----------+----------+--------------+------

In [49]:
df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- turbine_id: integer (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- wind_direction: integer (nullable = true)
 |-- power_output: double (nullable = true)



In [50]:
#Data Quality Validation (Null Check Verification)
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+---------+----------+----------+--------------+------------+
|timestamp|turbine_id|wind_speed|wind_direction|power_output|
+---------+----------+----------+--------------+------------+
|        0|         0|         0|             0|           0|
+---------+----------+----------+--------------+------------+



In [51]:
#Deduplication and Record Count Verification
df = df.dropDuplicates()

print("Rows after removing duplicates:")
print(df.count())

Rows after removing duplicates:
11160


In [52]:
df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- turbine_id: integer (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- wind_direction: integer (nullable = true)
 |-- power_output: double (nullable = true)



In [53]:
df.show(5, truncate=False)

+-------------------+----------+----------+--------------+------------+
|timestamp          |turbine_id|wind_speed|wind_direction|power_output|
+-------------------+----------+----------+--------------+------------+
|2022-03-04 20:00:00|11        |12.6      |300           |3.5         |
|2022-03-06 08:00:00|12        |9.2       |314           |3.4         |
|2022-03-06 09:00:00|12        |14.7      |255           |3.8         |
|2022-03-11 07:00:00|14        |12.1      |206           |4.3         |
|2022-03-11 08:00:00|12        |13.8      |231           |1.8         |
+-------------------+----------+----------+--------------+------------+
only showing top 5 rows


In [54]:
#Null Value Assessment
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+---------+----------+----------+--------------+------------+
|timestamp|turbine_id|wind_speed|wind_direction|power_output|
+---------+----------+----------+--------------+------------+
|        0|         0|         0|             0|           0|
+---------+----------+----------+--------------+------------+



In [55]:
clean_df = df.na.drop(subset=["power_output"])

In [56]:
clean_df = clean_df.dropDuplicates()

print("Rows after removing duplicates:", clean_df.count())

Rows after removing duplicates: 11160


In [57]:
# Duplicate Identification
duplicates = df.groupBy("timestamp", "turbine_id").count().filter("count > 1")

print("Duplicate Records:")
duplicates.show()

print("Duplicate Count:", duplicates.count())

Duplicate Records:
+---------+----------+-----+
|timestamp|turbine_id|count|
+---------+----------+-----+
+---------+----------+-----+

Duplicate Count: 0


In [58]:
# Calculating daily power aggregates grouped by date and turbine ID. 24 hours daily aggregation
import pyspark.sql.functions as F
summary = clean_df.withColumn("date", F.to_date(F.col("timestamp"))).groupBy("date", "turbine_id").agg(
    F.round(F.min("power_output"), 2).alias("min_power_mw"),
    F.round(F.max("power_output"), 2).alias("max_power_mw"),
    F.round(F.avg("power_output"), 2).alias("avg_power_mw"),
    F.count("power_output").alias("sample_count")
)

summary.show(10)

+----------+----------+------------+------------+------------+------------+
|      date|turbine_id|min_power_mw|max_power_mw|avg_power_mw|sample_count|
+----------+----------+------------+------------+------------+------------+
|2022-03-31|        15|         1.7|         4.4|        2.92|          24|
|2022-03-14|         6|         1.7|         4.4|        3.24|          24|
|2022-03-11|         7|         1.6|         4.5|        3.18|          24|
|2022-03-18|         7|         1.6|         4.4|        3.02|          24|
|2022-03-13|        11|         1.5|         4.5|         3.0|          24|
|2022-03-08|         2|         1.6|         4.4|        3.08|          24|
|2022-03-29|         1|         1.5|         4.5|        3.13|          24|
|2022-03-10|        10|         1.5|         4.5|        3.04|          24|
|2022-03-30|         9|         1.5|         4.2|        2.83|          24|
|2022-03-19|         2|         1.5|         4.4|        2.79|          24|
+----------+

In [59]:
# Mean and Standard Deviation
from pyspark.sql.functions import stddev

stats = clean_df.groupBy("turbine_id").agg(
    avg("power_output").alias("mean_power"),
    stddev("power_output").alias("std_power")
)

In [60]:
# Statistical Baseline Joining
joined = clean_df.join(stats, on="turbine_id")

In [61]:
# Statistical Anomaly Detection
from pyspark.sql.functions import col

anomalies = joined.filter(
    (col("power_output") > col("mean_power") + 2 * col("std_power")) |
    (col("power_output") < col("mean_power") - 2 * col("std_power"))
)

anomalies.show()

+----------+---------+----------+--------------+------------+----------+---------+
|turbine_id|timestamp|wind_speed|wind_direction|power_output|mean_power|std_power|
+----------+---------+----------+--------------+------------+----------+---------+
+----------+---------+----------+--------------+------------+----------+---------+



In [62]:
clean_df.write.mode("overwrite").option("header", True).csv("cleaned_data")

In [63]:
summary.write.mode("overwrite").option("header", True).csv("summary_statistics")

In [64]:
anomalies.write.mode("overwrite").option("header", True).csv("anomalies")

In [65]:
# Outlier Removal and Data Cleaning
from pyspark.sql.functions import avg, stddev, col

# Calculate mean and standard deviation for each turbine
stats = clean_df.groupBy("turbine_id").agg(
    avg("power_output").alias("mean_power"),
    stddev("power_output").alias("std_power")
)

# Join statistics back
clean_df = clean_df.join(stats, "turbine_id")

# Remove values outside ±3 standard deviations
clean_df = clean_df.filter(
    (col("power_output") >= col("mean_power") - 3 * col("std_power")) &
    (col("power_output") <= col("mean_power") + 3 * col("std_power"))
)

# Drop helper columns
clean_df = clean_df.drop("mean_power", "std_power")

print("Rows after removing outliers:", clean_df.count())

Rows after removing outliers: 11160


In [66]:
# Turbine Summary Statistics Calculation
summary = df.groupBy("turbine_id").agg(
    min("power_output").alias("Minimum"),
    max("power_output").alias("Maximum"),
    avg("power_output").alias("Average"),
    stddev("power_output").alias("StdDev"),
    count("*").alias("Measurements")
)

summary.show()

+----------+-------+-------+------------------+------------------+------------+
|turbine_id|Minimum|Maximum|           Average|            StdDev|Measurements|
+----------+-------+-------+------------------+------------------+------------+
|        12|    1.5|    4.5| 3.051209677419357|0.8610761298124979|         744|
|         1|    1.5|    4.5|3.0163978494623653|0.8572389449056389|         744|
|        13|    1.5|    4.5| 3.031317204301078|0.8773202723151072|         744|
|         6|    1.5|    4.5|2.9838709677419377| 0.874112856646467|         744|
|         3|    1.5|    4.5| 2.979973118279573|0.8626320481064979|         744|
|         5|    1.5|    4.5| 3.016532258064512|0.8658829219297749|         744|
|        15|    1.5|    4.5| 3.036424731182797|0.8537802013874648|         744|
|         9|    1.5|    4.5|  3.00282258064516|0.8742649845618986|         744|
|         4|    1.5|    4.5|2.9463709677419376|0.8870309797871536|         744|
|         8|    1.5|    4.5| 2.984811827

In [67]:
# Persistence to SQLite Database
import sqlite3

# Convert Spark DataFrames to Pandas
clean_pd = clean_df.toPandas()
summary_pd = summary.toPandas()
anomaly_pd = anomalies.toPandas()

# Create SQLite database
conn = sqlite3.connect("wind_turbine.db")

clean_pd.to_sql("cleaned_data", conn, if_exists="replace", index=False)
summary_pd.to_sql("summary_statistics", conn, if_exists="replace", index=False)
anomaly_pd.to_sql("anomalies", conn, if_exists="replace", index=False)

conn.close()

print("Database created successfully!")

Database created successfully!


In [68]:
from google.colab import files

files.download("wind_turbine.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [69]:
# Data Archiving and Export Compression
import shutil

shutil.make_archive(
    "Wind_Turbine_Output",
    "zip",
    root_dir=".",
    base_dir="cleaned_data"
)

'/content/Wind_Turbine_Output.zip'

In [70]:
# Final Output Directory Consolidation & Organization
import os
import shutil

os.makedirs("final_output", exist_ok=True)

shutil.copytree("cleaned_data", "final_output/cleaned_data", dirs_exist_ok=True)
shutil.copytree("summary_statistics", "final_output/summary_statistics", dirs_exist_ok=True)
shutil.copytree("anomalies", "final_output/anomalies", dirs_exist_ok=True)

'final_output/anomalies'

In [71]:
shutil.make_archive("Wind_Turbine_Output", "zip", "final_output")

'/content/Wind_Turbine_Output.zip'

In [72]:
from google.colab import files

files.download("Wind_Turbine_Output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>